# MLP Session 25 - OPPE 1 Practice Sep'25

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Data Preprocessing

In [ ]:
df = pd.read_csv(
    "/kaggle/input/mlp-oppe-1-sep-25/kaggle_assignment_comp_train.csv",
    index_col="id"
)
df.head()

### Q1: How many samples are there in the dataset?

In [ ]:
df.shape[0]

### Q2: How many unique locations are present in the dataset?

In [ ]:
df["location"].nunique()

In [ ]:
len(df["location"].unique()) - 1

### Q3: What is the average house price (in lakhs)?

In [ ]:
df["price"].mean()

### Q4: How many houses have more than 3 bathrooms?

In [ ]:
df.loc[df.bath > 3].shape[0]

In [ ]:
(df.bath > 3).sum()

### Q5: What is the average price of the top 10 most expensive houses?

In [ ]:
# average price
# top 10 most expensive
df["price"].nlargest(10).mean()

In [ ]:
df['price'].sort_values(ascending=False).head(10).mean()

In [ ]:
df['price'].sort_values().tail(10).mean()

### Q6: How many missing values exist in the total_sqft feature?

In [ ]:
df["total_sqft"].isnull().sum()

In [ ]:
df.info()

### Q7: How many houses have 3 bedrooms and at least 2 balconies?

In [ ]:
df["size"].unique()

In [ ]:
# 3 bedrooms and
# df["size"].str.split(" ").apply(lambda x: x[0] if type(x)==list else 0)=="3"




# >=2 balconies
df[
((df["size"]=="3 BHK") | (df["size"]=="3 Bedrooms")) & 
(df["balcony"] > 1)
].shape

# df[
# (df["size"].str.contains("3")) & 
# (df["balcony"] > 1)
# ].shape

### Q8: What is the average price per square foot?

In [ ]:
df2 = df.dropna(subset=["total_sqft"])
df2["price"].sum() / df2["total_sqft"].sum()

Replace missing total_sqft values with the median of the column.
### Q9: After imputation, what is the new count of missing values?

In [ ]:
median_sqft = df["total_sqft"].median()

df["total_sqft"].fillna(median_sqft).isna().sum()
df["total_sqft"].fillna(median_sqft).mean()

In [ ]:
from sklearn.impute import SimpleImputer

- fit - computes the value using the strategy from training dataset - applies on training dataset
- transform - impute the missing values with the computed value - applies on test dataset
- fit_transform - computes & impute the missing values... - applies on training dataset


- strategies - mean, median, most_frequent, constant

In [ ]:
imputer = SimpleImputer(strategy="median")
imputer.fit_transform(df[["total_sqft"]]).mean()

Encode the area_type column using one-hot encoding.
### Q10: How many new columns are created?

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df["area_type"].unique()

In [ ]:
df["area_type"]

In [ ]:
ohe = OneHotEncoder(sparse=False)
ohe.fit_transform(df[["area_type"]])

Scale price and total_sqft using MinMaxScaler.
### Q11: What is the mean of the scaled price column?

In [ ]:
from sklearn.preprocessing import MinMaxScaler

mm_scaler = MinMaxScaler()
mm_scaler.fit_transform(df[["price", "total_sqft"]])[:, 0].mean()

Drop rows with missing values in bath or balcony.
### Q12: How many rows remain?

In [ ]:
df.dropna(subset=["bath", "balcony"]).shape

Split the dataset into 70% train and 30% test.
### Q13: How many samples are in the training set?

In [ ]:
X = df.iloc[:, :-1]
y = df["price"]

X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=37)
X_train.shape, X_test.shape

Create a new categorical column `PRICE_CATEGORY` based on price:

- <50 → “Low”

- 50–100 → “Medium”

- 100 → “High”
### Q14: Which category has the most records?

In [ ]:
def categorize_price(price):
    if price < 50:
        return "Low"
    elif 50 <= price < 100:
        return "Medium"
    return "High"

df["PRICE_CATEGORY"] = df["price"].apply(categorize_price)
df.head()

In [ ]:
df.drop(columns=["PRICE_CATEGORY"], inplace=True)

## Model Building

Split the dataset into train/test (test_size=0.3, random_state=42).

### Q1: How many samples are there in training set?

#### category columns - fill missing with most_frequent
#### category columns - OrdinalEncoding
#### numerical columns - fill missing with mean


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape

In [ ]:
num_cols = X.select_dtypes(exclude="object").columns

cat_cols = X.select_dtypes(include="object").columns

num_cols, cat_cols

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

cat_pipeline

In [ ]:
transformer = ColumnTransformer(transformers=[
    ("num", SimpleImputer(), num_cols),
    ("cat", cat_pipeline, cat_cols)
], remainder="passthrough")
transformer
transformer.set_output(transform="pandas")

In [ ]:
X_train_tf = transformer.fit_transform(X_train)
X_test_tf = transformer.transform(X_test)

In [ ]:
X_test_tf

#### category columns - do one hot encoding on columns having < 5 unique values, and ordinalencoding on >= 5 unique values

Train a Ridge Regression model (α=10, solver='saga').
### Q2: What is the R² score on test data?

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=10, solver="saga", random_state=42)
ridge.fit(X_train_tf, y_train)

In [ ]:
ridge.score(X_test_tf, y_test)

### Q3: Which feature index is most important (largest coefficient)?

In [ ]:
ridge.coef_

In [ ]:
ridge.coef_.argmax()

### Q4: Which feature index is least important (smallest coefficient)?

In [ ]:
ridge.coef_.argmin()

Perform GridSearchCV with SGDRegressor.
Parameters:

penalty = ['l1', 'l2']

alpha = [1e-5, 1e-4, 1e-3]

tol = [1e-4, 1e-3]

### Q5: What is the best penalty?

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

In [ ]:
params = {
    "penalty": ['l1', 'l2'],
    "alpha": [1e-5, 1e-4, 1e-3],
    "tol": [1e-4, 1e-3]
}

gscv = GridSearchCV(SGDRegressor(random_state=42), params, cv=5)
gscv.fit(X_train_tf, y_train)

In [ ]:
gscv.best_params_

### Q6: What is the mean absolute error on test data using the best SGD model?

In [ ]:
y_pred = gscv.best_estimator_.predict(X_test_tf)
mean_absolute_error(y_test, y_pred)

Create a PCA + Lasso pipeline with GridSearchCV.
Parameters:

n_components = [0.9, 0.95]

alpha = [10, 1, 0.01]

### Q7: What is the best alpha value?

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.decomposition import PCA

In [ ]:
pca_lasso = Pipeline([
    ("pca", PCA()),
    ("lasso", Lasso())
])
pca_lasso

In [ ]:
params = {
    "pca__n_components": [0.9, 0.95],
    "lasso__alpha": [10, 1, 0.01]
}

gscvlasso = GridSearchCV(pca_lasso, params, cv=5, n_jobs=-1)
gscvlasso.fit(X_train_tf, y_train)

In [ ]:
gscvlasso.best_params_

### Q8: What is the R² score of the best PCA+Lasso pipeline on the test set?

In [ ]:
gscvlasso.best_estimator_.score(X_test_tf, y_test)

### Q9: How much variance is explained by the first principal component?

In [ ]:
gscvlasso.best_estimator_.named_steps["pca"].explained_variance_ratio_

Create a PolynomialFeatures + Ridge pipeline (degree=2, alpha=1).
### Q10: What is the R² score on test data?

Apply Recursive Feature Elimination (RFE) using LinearRegression.
### Q11: Which feature is eliminated first?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE

rfe = RFE(LinearRegression(), n_features_to_select=6)
rfe.fit(X_train_tf, y_train)

In [ ]:
rfe.support_.argmin()

In [ ]:
X.columns[5]

### Q12: Train Lasso (α=0.1) and compute mean squared error on test data.

Train Ridge (α=1) and Lasso (α=1) and compare R² scores.
### Q13: Which performs better?

### Q14: Build a pipeline combining StandardScaler + Ridge (α=5) and compute MSE.